# Train the live CTAG patch predictor
This notebook is restart-safe. Select a **T4 GPU**, choose **Runtime → Run all**, and rerun it after any Colab disconnect. Completed data shards and optimizer checkpoints are reused from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROFILE = 'balanced'  # smoke, balanced, or quality
WORKSPACE = '/content/drive/MyDrive/ctag-live'
DEMO_PROMPT = 'a deep resonant train horn'

In [ ]:
from pathlib import Path
import os, shutil, subprocess, urllib.request
content = Path('/content')
repo = content / 'Synthesizer'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    cloned = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nikhilanayak/Synthesizer.git', str(repo)])
    if cloned.returncode:
        # Public archive fallback avoids broken Colab Git credential helpers.
        archive = content / 'Synthesizer-main.zip'
        urllib.request.urlretrieve('https://codeload.github.com/nikhilanayak/Synthesizer/zip/refs/heads/main', archive)
        shutil.unpack_archive(archive, content)
        repo = content / 'Synthesizer-main'
os.chdir(repo / 'ctag-repro')
print(Path.cwd())
!bash tools/colab_bootstrap.sh
!ctag doctor --require-gpu

## Resumable data generation and training
The balanced profile creates the random patch corpus, mines all 527 AudioSet labels, runs stronger CTAG teachers for the paper prompt set, trains both networks, and performs three real-CLAP improvement rounds.

In [ ]:
!ctag train-live --profile {PROFILE} --workspace {WORKSPACE} --device cuda

## Held-out evaluation and listening artifacts

In [ ]:
MODEL = f'{WORKSPACE}/training/model'
!ctag evaluate-direct --bundle {MODEL} --output {WORKSPACE}/evaluation --device cuda
import json
report = json.load(open(f'{WORKSPACE}/evaluation/report.json'))
print({k: v for k, v in report.items() if k != 'results'})

## Strict no-search live inference

In [ ]:
from ctag_repro.direct_training import LiveDirectSynth
live = LiveDirectSynth(MODEL, 'checkpoints/630k-audioset-best.pt', 'cuda')
result = live.generate(DEMO_PROMPT, f'{WORKSPACE}/demo')
print(result['timings_seconds'])
from IPython.display import Audio, display
from pathlib import Path
display(Audio(result['wav']))
# Try another prompt without reloading either model:
# result = live.generate('metallic church bells', f'{WORKSPACE}/demo')

## Hardware-facing ONNX and INT8 exports

In [ ]:
!ctag export-direct --bundle {MODEL} --output {WORKSPACE}/export --quantize
!cat {WORKSPACE}/export/export.json